# Lab 5C: Heuristic Alpha-Beta Tree Search in Connect Four

## Learning goals
By the end of this lab, students can:
- explain why full minimax is not feasible for larger games;
- implement a depth cutoff and a heuristic evaluation function;
- use alpha-beta pruning with move ordering in a nontrivial board game;
- visualize a partial search tree and the current board at each visited node;
- compare depth limits, node counts, cutoffs, and chosen moves.

Source note: This lab implements the Chapter 5 idea of replacing terminal utility with `EVAL` at a cutoff depth.

## Chapter 5 notes for this lab

For complex games, searching the full tree is too expensive. Heuristic alpha-beta search changes minimax in two ways:

1. `IS-CUTOFF(state, depth)` stops search at terminal states or at a selected depth limit.
2. `EVAL(state, player)` estimates how good a nonterminal state is.

A good evaluation function should be fast and correlated with winning chances. In this lab, Connect Four features include center control, open two-in-a-row threats, open three-in-a-row threats, immediate wins, and immediate blocks.

In [ ]:
# Run this cell first.
# %pip install matplotlib ipywidgets

import math
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Static first/last frames will still work.")

ROWS, COLS = 6, 7
MAX_PLAYER, MIN_PLAYER, EMPTY = 1, -1, 0


def make_stepper(steps, draw_func, info_func=None, title="Search stepper"):
    if not steps:
        print("No steps to display.")
        return
    if WIDGETS_AVAILABLE:
        slider = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, description="step")
        play = widgets.Play(value=0, min=0, max=len(steps)-1, step=1, interval=500)
        widgets.jslink((play, "value"), (slider, "value"))
        out = widgets.Output()
        def render(change=None):
            with out:
                clear_output(wait=True)
                display(HTML(f"<h3>{title}</h3>"))
                if info_func:
                    display(HTML(info_func(steps[slider.value], slider.value)))
                draw_func(steps[slider.value], slider.value)
                plt.show()
        slider.observe(render, names="value")
        display(widgets.VBox([widgets.HBox([play, slider]), out]))
        render()
    else:
        for i in [0, len(steps)-1]:
            if info_func:
                display(HTML(info_func(steps[i], i)))
            draw_func(steps[i], i)
            plt.show()


def new_board():
    return tuple(tuple(EMPTY for _ in range(COLS)) for _ in range(ROWS))


def valid_moves(board):
    return [c for c in range(COLS) if board[0][c] == EMPTY]


def drop_piece(board, col, player):
    if board[0][col] != EMPTY:
        raise ValueError(f"Column {col} is full.")
    b = [list(row) for row in board]
    for r in range(ROWS-1, -1, -1):
        if b[r][col] == EMPTY:
            b[r][col] = player
            return tuple(tuple(row) for row in b)
    raise ValueError("Column is full.")


def board_after_moves(moves):
    board = new_board()
    player = MAX_PLAYER
    for col in moves:
        board = drop_piece(board, col, player)
        player *= -1
    return board, player


def check_winner(board):
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for r in range(ROWS):
        for c in range(COLS):
            player = board[r][c]
            if player == EMPTY:
                continue
            for dr, dc in directions:
                cells = []
                for k in range(4):
                    rr, cc = r + dr*k, c + dc*k
                    if 0 <= rr < ROWS and 0 <= cc < COLS:
                        cells.append(board[rr][cc])
                if len(cells) == 4 and all(x == player for x in cells):
                    return player
    return None


def is_terminal(board):
    return check_winner(board) is not None or not valid_moves(board)


def all_windows(board):
    windows = []
    for r in range(ROWS):
        for c in range(COLS-3):
            windows.append([board[r][c+i] for i in range(4)])
    for c in range(COLS):
        for r in range(ROWS-3):
            windows.append([board[r+i][c] for i in range(4)])
    for r in range(ROWS-3):
        for c in range(COLS-3):
            windows.append([board[r+i][c+i] for i in range(4)])
    for r in range(3, ROWS):
        for c in range(COLS-3):
            windows.append([board[r-i][c+i] for i in range(4)])
    return windows

In [ ]:
def score_window(window, player=MAX_PLAYER):
    """Feature score from MAX's perspective."""
    opp = -player
    p_count = window.count(player)
    o_count = window.count(opp)
    e_count = window.count(EMPTY)
    if p_count == 4:
        return 100000
    if o_count == 4:
        return -100000
    score = 0
    if p_count == 3 and e_count == 1:
        score += 120
    elif p_count == 2 and e_count == 2:
        score += 12
    if o_count == 3 and e_count == 1:
        score -= 150  # blocking an opponent three is urgent
    elif o_count == 2 and e_count == 2:
        score -= 10
    return score


def evaluate_board(board, player=MAX_PLAYER):
    """Fast heuristic EVAL(board, MAX). Higher is better for MAX."""
    winner = check_winner(board)
    if winner == MAX_PLAYER:
        return 1_000_000
    if winner == MIN_PLAYER:
        return -1_000_000
    score = 0
    center_col = [board[r][COLS//2] for r in range(ROWS)]
    score += 5 * center_col.count(MAX_PLAYER)
    score -= 5 * center_col.count(MIN_PLAYER)
    for window in all_windows(board):
        score += score_window(window, MAX_PLAYER)
    return score


def ordered_moves(board, player, use_ordering=True):
    moves = valid_moves(board)
    if not use_ordering:
        return moves
    center_bias = {c: -abs(c - COLS//2) for c in moves}
    scored = []
    for c in moves:
        nb = drop_piece(board, c, player)
        scored.append((evaluate_board(nb), center_bias[c], c))
    scored.sort(reverse=(player == MAX_PLAYER))
    return [c for _, _, c in scored]


def draw_connect4_board(ax, board, title="", last_col=None):
    ax.set_aspect("equal")
    ax.set_xlim(-0.5, COLS-0.5)
    ax.set_ylim(-0.5, ROWS-0.5)
    ax.axis("off")
    ax.set_title(title)
    ax.add_patch(Rectangle((-0.5, -0.5), COLS, ROWS, facecolor="#1f77b4", edgecolor="black", lw=2))
    for r in range(ROWS):
        for c in range(COLS):
            val = board[r][c]
            y = ROWS - 1 - r
            color = "white"
            if val == MAX_PLAYER:
                color = "#ffd23f"
            elif val == MIN_PLAYER:
                color = "#d00000"
            ax.add_patch(Circle((c, y), 0.38, facecolor=color, edgecolor="black", lw=1.2))
    if last_col is not None:
        ax.add_patch(Rectangle((last_col-0.48, -0.48), 0.96, ROWS-0.04, fill=False, edgecolor="limegreen", lw=3))
    ax.text(0, -0.9, "MAX = yellow", fontsize=9, ha="left")
    ax.text(3.8, -0.9, "MIN = red", fontsize=9, ha="left")

In [ ]:
def alpha_beta_search(board, depth_limit=4, player=MAX_PLAYER, use_ordering=True, use_tt=True, record=False, visual_depth=2):
    """Heuristic alpha-beta with cutoff and optional exact transposition cache."""
    trace = []
    stats = {"nodes": 0, "cutoffs": 0, "tt_hits": 0}
    tt = {}

    def search(b, depth, alpha, beta, to_move, path=()):
        stats["nodes"] += 1
        if record and len(path) <= visual_depth:
            trace.append({"kind": "enter", "path": path, "board": b, "depth_left": depth, "player": to_move, "alpha": alpha, "beta": beta,
                          "message": f"Enter path {path if path else 'root'}; depth left={depth}; player={'MAX' if to_move==MAX_PLAYER else 'MIN'}."})
        key = (b, depth, to_move)
        if use_tt and key in tt:
            stats["tt_hits"] += 1
            value, move = tt[key]
            if record and len(path) <= visual_depth:
                trace.append({"kind": "tt", "path": path, "board": b, "value": value, "best_move": move,
                              "message": f"Transposition-table hit at path {path}: value={value}."})
            return value, move, True
        winner = check_winner(b)
        if winner is not None or not valid_moves(b) or depth == 0:
            value = evaluate_board(b)
            if record and len(path) <= visual_depth:
                reason = "terminal" if winner is not None or not valid_moves(b) else "cutoff depth"
                trace.append({"kind": "eval", "path": path, "board": b, "value": value,
                              "message": f"Evaluate {reason} at path {path}: EVAL={value}."})
            return value, None, True

        complete = True
        moves = ordered_moves(b, to_move, use_ordering=use_ordering)
        if to_move == MAX_PLAYER:
            value, best_move = -math.inf, None
            for i, move in enumerate(moves):
                child = drop_piece(b, move, to_move)
                child_value, _, child_complete = search(child, depth-1, alpha, beta, -to_move, path + (move,))
                if child_value > value:
                    value, best_move = child_value, move
                alpha = max(alpha, value)
                if record and len(path) <= visual_depth:
                    trace.append({"kind": "update", "path": path, "board": b, "value": value, "best_move": best_move,
                                  "alpha": alpha, "beta": beta,
                                  "message": f"MAX updates path {path if path else 'root'}: best column={best_move}, value={value}, alpha={alpha}."})
                if value >= beta:
                    stats["cutoffs"] += 1
                    complete = False
                    pruned = [path + (m,) for m in moves[i+1:]]
                    if record and len(path) <= visual_depth:
                        trace.append({"kind": "prune", "path": path, "board": b, "pruned_paths": pruned,
                                      "message": f"MAX prunes remaining moves {moves[i+1:]} because value {value} >= beta {beta}."})
                    break
        else:
            value, best_move = math.inf, None
            for i, move in enumerate(moves):
                child = drop_piece(b, move, to_move)
                child_value, _, child_complete = search(child, depth-1, alpha, beta, -to_move, path + (move,))
                if child_value < value:
                    value, best_move = child_value, move
                beta = min(beta, value)
                if record and len(path) <= visual_depth:
                    trace.append({"kind": "update", "path": path, "board": b, "value": value, "best_move": best_move,
                                  "alpha": alpha, "beta": beta,
                                  "message": f"MIN updates path {path if path else 'root'}: best column={best_move}, value={value}, beta={beta}."})
                if value <= alpha:
                    stats["cutoffs"] += 1
                    complete = False
                    pruned = [path + (m,) for m in moves[i+1:]]
                    if record and len(path) <= visual_depth:
                        trace.append({"kind": "prune", "path": path, "board": b, "pruned_paths": pruned,
                                      "message": f"MIN prunes remaining moves {moves[i+1:]} because value {value} <= alpha {alpha}."})
                    break
        if use_tt and complete:
            tt[key] = (value, best_move)
        if record and len(path) <= visual_depth:
            trace.append({"kind": "return", "path": path, "board": b, "value": value, "best_move": best_move,
                          "message": f"Return value={value}, best column={best_move} from path {path if path else 'root'}."})
        return value, best_move, complete

    value, move, _ = search(board, depth_limit, -math.inf, math.inf, player, ())
    return {"value": value, "move": move, "stats": stats, "trace": trace}

In [ ]:
def build_path_tree(board, max_depth=2, player=MAX_PLAYER):
    nodes, edges = [], []
    def rec(b, path, depth, to_move, parent_id=None, move=None):
        node_id = len(nodes)
        nodes.append({"id": node_id, "path": path, "board": b, "depth": depth, "player": to_move, "move": move})
        if parent_id is not None:
            edges.append((parent_id, node_id, move))
        if depth < max_depth and not is_terminal(b):
            for m in valid_moves(b):
                rec(drop_piece(b, m, to_move), path + (m,), depth+1, -to_move, node_id, m)
        return node_id
    rec(board, (), 0, player)
    children = defaultdict(list)
    for p, c, _ in edges:
        children[p].append(c)
    x_counter = [0]
    pos = {}
    def assign(nid):
        if not children[nid]:
            x = x_counter[0]
            x_counter[0] += 1
        else:
            xs = [assign(ch) for ch in children[nid]]
            x = sum(xs) / len(xs)
        pos[nid] = (x, -nodes[nid]["depth"])
        return x
    assign(0)
    return nodes, edges, pos


def draw_connect4_tree(ax, root_board, trace, step_idx, visual_depth=2):
    nodes, edges, pos = build_path_tree(root_board, max_depth=visual_depth, player=MAX_PLAYER)
    node_by_id = {n["id"]: n for n in nodes}
    prefix = trace[:step_idx+1]
    seen = {()}
    returned = {}
    pruned = set()
    for ev in prefix:
        seen.add(ev.get("path", ()))
        if ev["kind"] in ("eval", "return", "tt"):
            returned[ev["path"]] = ev.get("value")
        if ev["kind"] == "prune":
            pruned.update(ev.get("pruned_paths", []))
    current_path = trace[step_idx].get("path", ())
    for p, c, move in edges:
        x1, y1 = pos[p]
        x2, y2 = pos[c]
        child_path = node_by_id[c]["path"]
        color = "#bbbbbb" if child_path in pruned else ("#444444" if child_path in seen else "#dddddd")
        style = "--" if child_path in pruned else "-"
        ax.plot([x1, x2], [y1, y2], color=color, ls=style, lw=1.5)
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.05, str(move), fontsize=8, ha="center")
    for n in nodes:
        x, y = pos[n["id"]]
        path = n["path"]
        if path == current_path:
            face, edge, lw = "#ffbf66", "#111111", 2.2
        elif path in pruned:
            face, edge, lw = "#eeeeee", "#999999", 0.8
        elif path in returned:
            face, edge, lw = "#b6e3a8", "#333333", 1.2
        elif path in seen:
            face, edge, lw = "#d7e9ff", "#333333", 1.0
        else:
            face, edge, lw = "#f7f7f7", "#aaaaaa", 0.8
        ax.add_patch(Circle((x, y), 0.22, facecolor=face, edgecolor=edge, lw=lw))
        label = "root" if path == () else str(path[-1])
        if path in returned:
            label += f"\n{returned[path]:.0f}"
        ax.text(x, y, label, ha="center", va="center", fontsize=7)
    ax.axis("off")
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs)-0.8, max(xs)+0.8)
    ax.set_ylim(min(ys)-0.6, max(ys)+0.6)
    ax.set_title(f"Visual tree: first {visual_depth} ply")


def root_action_values(board, depth):
    values = []
    for c in valid_moves(board):
        nb = drop_piece(board, c, MAX_PLAYER)
        result = alpha_beta_search(nb, depth_limit=max(depth-1, 0), player=MIN_PLAYER, use_ordering=True, use_tt=True)
        values.append((c, result["value"]))
    return values


def draw_root_bars(ax, values, selected=None):
    cols = [c for c, _ in values]
    vals = [v for _, v in values]
    colors = ["#ffbf66" if c == selected else "#8ecae6" for c in cols]
    ax.bar([str(c) for c in cols], vals, color=colors, edgecolor="black")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel("candidate column")
    ax.set_ylabel("heuristic alpha-beta value")
    ax.set_title("Root action values")
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.0f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=8)


def draw_c4_snapshot(ev, idx):
    board = ev.get("board", DEMO_BOARD)
    selected = ev.get("best_move", None)
    fig = plt.figure(figsize=(16, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 2.5, 1.35])
    ax_board = fig.add_subplot(gs[0, 0])
    ax_tree = fig.add_subplot(gs[0, 1])
    ax_bar = fig.add_subplot(gs[0, 2])
    draw_connect4_board(ax_board, board, title="Current search node", last_col=selected)
    draw_connect4_tree(ax_tree, DEMO_BOARD, TRACE, idx, visual_depth=2)
    draw_root_bars(ax_bar, ROOT_VALUES, selected=selected if ev.get("path", ()) == () else None)
    fig.tight_layout()


def c4_info_html(ev, idx):
    return f"""
    <div style='font-family:Arial;line-height:1.35'>
    <b>Step {idx+1} of {len(TRACE)}</b> &nbsp; <b>Event:</b> {ev['kind']}<br>
    {ev['message']}<br>
    Nodes are colored when entered, evaluated, returned, or pruned. Values are from MAX/yellow's viewpoint.
    </div>
    """

In [ ]:
# A midgame position that is still small enough to visualize.
DEMO_MOVES = [3, 2, 3, 2, 4, 1, 4, 5, 5, 6]
DEMO_BOARD, PLAYER_TO_MOVE = board_after_moves(DEMO_MOVES)

SEARCH_DEPTH = 4
RESULT = alpha_beta_search(DEMO_BOARD, depth_limit=SEARCH_DEPTH, player=PLAYER_TO_MOVE, use_ordering=True, use_tt=True, record=True, visual_depth=2)
TRACE = RESULT["trace"]
ROOT_VALUES = root_action_values(DEMO_BOARD, SEARCH_DEPTH)

print("Player to move:", "MAX/yellow" if PLAYER_TO_MOVE == MAX_PLAYER else "MIN/red")
print("Recommended column:", RESULT["move"])
print("Search value:", RESULT["value"])
print("Stats:", RESULT["stats"])

make_stepper(TRACE, draw_c4_snapshot, c4_info_html, title="Heuristic alpha-beta Connect Four search")

## Depth, ordering, and node counts

The next comparison shows why a fixed depth limit is practical only when combined with pruning and ordering. The heuristic value is approximate; deeper search is usually stronger but costs more.

In [ ]:
def compare_depths(board, player, max_depth=5):
    rows = []
    for depth in range(1, max_depth+1):
        ordered = alpha_beta_search(board, depth_limit=depth, player=player, use_ordering=True, use_tt=True)
        unordered = alpha_beta_search(board, depth_limit=depth, player=player, use_ordering=False, use_tt=False)
        rows.append({
            "depth": depth,
            "ordered_nodes": ordered["stats"]["nodes"],
            "unordered_nodes": unordered["stats"]["nodes"],
            "ordered_cutoffs": ordered["stats"]["cutoffs"],
            "unordered_cutoffs": unordered["stats"]["cutoffs"],
            "ordered_move": ordered["move"],
            "ordered_value": ordered["value"],
        })
    return rows

rows = compare_depths(DEMO_BOARD, PLAYER_TO_MOVE, max_depth=5)
for row in rows:
    print(row)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r["depth"] for r in rows], [r["unordered_nodes"] for r in rows], marker="o", label="No move ordering")
ax.plot([r["depth"] for r in rows], [r["ordered_nodes"] for r in rows], marker="o", label="Heuristic move ordering")
ax.set_yscale("log")
ax.set_xlabel("depth limit")
ax.set_ylabel("nodes visited, log scale")
ax.set_title("Heuristic alpha-beta cost by depth")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

## Student practice

Change the move list, depth limit, or evaluation weights. Then answer: does the recommended move change? Did deeper search increase confidence or reveal a tactical problem?

In [ ]:
STUDENT_MOVES = [3, 3, 2, 4, 2, 4, 1, 5]
student_board, student_player = board_after_moves(STUDENT_MOVES)
student_depth = 4
student_result = alpha_beta_search(student_board, depth_limit=student_depth, player=student_player, use_ordering=True, use_tt=True)

fig, ax = plt.subplots(figsize=(5, 4))
draw_connect4_board(ax, student_board, title=f"Student board; best column {student_result['move']}", last_col=student_result["move"])
plt.show()
print("Result:", student_result)
print("Root action values:", root_action_values(student_board, student_depth))